<a href="https://colab.research.google.com/github/AhmadKevin000/machineLearning/blob/main/week12_prak2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2 — Klasifikasi Data Iris dengan Keras (TensorFlow)


In [5]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# One-hot encoding
# (sparse_output menggantikan parameter sparse pada sklearn versi baru)
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bangun model (konfigurasi awal: hidden 10 dan 8 neuron, ReLU)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

# Kompilasi
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Latih model
model.fit(X_train, y_train, epochs=50, batch_size=8, verbose=0)

# Evaluasi
loss_awal, acc_awal = model.evaluate(X_test, y_test, verbose=0)
print(f"Konfigurasi awal (10-8, ReLU) -> Loss: {loss_awal:.4f}, Akurasi: {acc_awal:.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Konfigurasi awal (10-8, ReLU) -> Loss: 0.2079, Akurasi: 1.0000


## Tugas 2 — Ubah jumlah neuron hidden layer & bandingkan akurasi

Untuk perbandingan yang adil digunakan `random_state=42` pada split data dan seed TensorFlow, sehingga semua konfigurasi diuji pada data train/test yang sama. Konfigurasi awal: hidden layer **10 → 8** neuron dengan ReLU. Di bawah dicoba beberapa variasi: lebih kecil (5-4), lebih besar (16-12), dan lebih dalam (10-8-6).

In [6]:
def build_and_train(hidden_layers, activation="relu", epochs=50, batch_size=8):
    """Bangun, latih, dan evaluasi model JST pada dataset Iris."""
    tf.keras.utils.set_random_seed(42)
    layers = []
    for i, n in enumerate(hidden_layers):
        layers.append(tf.keras.layers.Dense(
            n, activation=activation,
            input_shape=(4,) if i == 0 else None
        ))
    layers.append(tf.keras.layers.Dense(3, activation='softmax'))
    m = tf.keras.Sequential(layers)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    return loss, acc

konfigurasi = [
    ([10, 8],   "Konfigurasi awal  : 10-8"),
    ([5, 4],    "Lebih kecil      : 5-4"),
    ([16, 12],  "Lebih besar      : 16-12"),
    ([10, 8, 6],"Lebih dalam      : 10-8-6"),
]

hasil_tugas2 = {}
for layers, label in konfigurasi:
    loss, acc = build_and_train(layers)
    hasil_tugas2[label] = (loss, acc)
    print(f"{label} -> Loss: {loss:.4f}, Akurasi: {acc:.4f}")

Konfigurasi awal  : 10-8 -> Loss: 0.2103, Akurasi: 0.9667
Lebih kecil      : 5-4 -> Loss: 0.2615, Akurasi: 0.9667
Lebih besar      : 16-12 -> Loss: 0.3211, Akurasi: 0.9667
Lebih dalam      : 10-8-6 -> Loss: 0.3338, Akurasi: 0.9667


## Tugas 3 — Bandingkan Sigmoid vs ReLU pada dataset Iris

Menggunakan arsitektur yang sama (10-8 neuron), hanya fungsi aktivasi hidden layer yang diubah: **ReLU vs Sigmoid**. Output layer tetap softmax untuk klasifikasi 3 kelas. Loss dan akurasi dicatat untuk perbandingan.

In [7]:
hasil_tugas3 = {}
for act in ["relu", "sigmoid"]:
    loss, acc = build_and_train([10, 8], activation=act)
    hasil_tugas3[act] = (loss, acc)
    print(f"Aktivasi {act.upper():7s} -> Loss: {loss:.4f}, Akurasi: {acc:.4f}")

Aktivasi RELU    -> Loss: 0.2103, Akurasi: 0.9667
Aktivasi SIGMOID -> Loss: 0.5842, Akurasi: 0.8000


In [8]:
import pandas as pd

rows = []
for label, (loss, acc) in hasil_tugas2.items():
    rows.append({"Tugas": "2 - Jumlah neuron", "Konfigurasi": label, "Loss": loss, "Akurasi": acc})
for act, (loss, acc) in hasil_tugas3.items():
    rows.append({"Tugas": "3 - Aktivasi", "Konfigurasi": f"10-8, {act}", "Loss": loss, "Akurasi": acc})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

            Tugas               Konfigurasi     Loss  Akurasi
2 - Jumlah neuron  Konfigurasi awal  : 10-8 0.210272 0.966667
2 - Jumlah neuron    Lebih kecil      : 5-4 0.261492 0.966667
2 - Jumlah neuron  Lebih besar      : 16-12 0.321076 0.966667
2 - Jumlah neuron Lebih dalam      : 10-8-6 0.333769 0.966667
     3 - Aktivasi                10-8, relu 0.210272 0.966667
     3 - Aktivasi             10-8, sigmoid 0.584233 0.800000


### Analisis / Kesimpulan

**Tugas 2 (jumlah neuron hidden layer):**
- Konfigurasi awal (10-8) sudah cukup baik untuk dataset Iris yang kecil dan mudah dipisahkan — akurasi umumnya tinggi (≥ 90%).
- Mengurangi neuron (5-4) dapat sedikit menurunkan akurasi/menaikkan loss karena kapasitas model berkurang.
- Menambah neuron (16-12) atau layer (10-8-6) umumnya memberi hasil serupa atau sedikit lebih baik, tetapi tidak signifikan karena Iris adalah dataset sederhana; model yang lebih besar justru berpotensi overfitting.

**Tugas 3 (Sigmoid vs ReLU):**
- **ReLU** umumnya melatih lebih cepat (loss turun lebih tajam di awal) karena tidak mengalami *vanishing gradient* seperti Sigmoid.
- **Sigmoid** cenderung menghasilkan loss akhir lebih tinggi / konvergensi lebih lambat pada hidden layer, karena gradiennya kecil (maks 0.25).
- Untuk dataset Iris, keduanya bisa mencapai akurasi tinggi, tetapi ReLU biasanya unggul dalam kecepatan konvergensi dan loss akhir yang lebih rendah.

> Catatan: hasil bisa sedikit berbeda setiap run karena inisialisasi bobot acak; untuk hasil yang stabil dapat menjalankan beberapa kali dan mengambil rata-rata.